# Import

In [1]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib as mpl
from scipy.sparse.linalg import eigsh
from scipy.linalg import cosm
from numpy.linalg import matrix_power

In [2]:
mpl.rcParams['figure.dpi'] = 400

mpl.rc('font', family='sans-serif', size=16)
mpl.rc('mathtext', fontset='cm')
mpl.rc('axes', linewidth=0.75, titlesize=16, labelsize=14, titlepad=10)
mpl.rc('xtick', top=True, bottom=True, labelsize=12, direction='in')
mpl.rc('xtick.major', size=5, width=0.75)
mpl.rc('xtick.minor', size=2, width=0.75, visible=True)
mpl.rc('ytick', right=True, left=True, labelsize=12, direction='in')
mpl.rc('ytick.major', size=5, width=0.75)
mpl.rc('ytick.minor', size=2, width=0.75, visible=True)
mpl.rc('legend', fontsize=12)

# Toy model

Single-mode extended GKP Hamiltonian in the harmonic-oscillator basis:

$$H = E_C(n - n_g)^2 + \frac{E_L}{2}\varphi^2 + E_{2J}\cos(2\varphi - 4\pi\Phi_\mathrm{ext}) - E_S\cos(2\pi n)$$

Parameters ($E_{2J}, E_S, E_C, E_L$) are the **extended GKP** effective parameters extracted from the multimode fit.

In [3]:
def toy_cos2phi_gkp_HO(const):
    """Build the toy extended-GKP Hamiltonian in a truncated HO basis."""
    key_list = ('Ec', 'El', 'Es', 'E2j', 'phi_ext', 'ng', 'ntrun')
    Ec, El, Es, E2j, phi_ext, ng, ntrun = [const.get(key) for key in key_list]

    phiZPF = (Ec / (El / 2)) ** (1 / 4)

    cre  = np.diag([np.sqrt(j) for j in range(1, ntrun)], -1)
    ani  = np.diag([np.sqrt(j) for j in range(1, ntrun)],  1)
    iden = np.eye(ntrun)

    phi_op = phiZPF / np.sqrt(2) * (cre + ani)
    n_op   = 1j / (np.sqrt(2) * phiZPF) * (cre - ani)

    kin = Ec * (matrix_power(n_op, 2) + 2 * ng * n_op) + El / 2 * matrix_power(phi_op, 2)
    pot = E2j * cosm(2 * phi_op - 2 * 2 * np.pi * phi_ext * iden) - Es * cosm(2 * np.pi * n_op)

    return kin + pot


def f01_toy(Ec, El, Es, E2j, phi_ext=0.0, ng=0.0, ntrun=800):
    """Return the 0→1 transition frequency (GHz) of the toy model."""
    H  = toy_cos2phi_gkp_HO(dict(Ec=Ec, El=El, Es=Es, E2j=E2j,
                                  phi_ext=phi_ext, ng=ng, ntrun=ntrun))
    ev = sorted(eigsh(H, k=2, which='SA', return_eigenvectors=False))
    return ev[1] - ev[0]

# Device A

| $E_{2J}$ | $E_S$ | $E_C$ | $E_L$ |
|----------|-------|-------|-------|
| 3.97     | 2.95  | 0.42  | 0.41  |

In [4]:
E2j_A, Es_A, Ec_A, El_A = 3.97, 2.95, 0.42, 0.41

# Charge dispersion: f01(ng=0) vs f01(ng=0.5) at phi_ext=0
w01_ng0  = f01_toy(Ec_A, El_A, Es_A, E2j_A, phi_ext=0.0, ng=0.0)
w01_ng05 = f01_toy(Ec_A, El_A, Es_A, E2j_A, phi_ext=0.0, ng=0.5)

# Flux dispersion: f01(phi_ext=0) vs f01(phi_ext=0.25) at ng=0
w01_phi0   = f01_toy(Ec_A, El_A, Es_A, E2j_A, phi_ext=0.0,  ng=0.0)
w01_phi025 = f01_toy(Ec_A, El_A, Es_A, E2j_A, phi_ext=0.25, ng=0.0)

print("── Device A (toy model) ───────────────────────────")
print(f"  f01 (ng=0):      {w01_ng0*1e3:.4f} MHz")
print(f"  f01 (ng=0.5):    {w01_ng05*1e3:.4f} MHz")
print(f"  Charge disp.:    {abs(w01_ng05 - w01_ng0)*1e6:.2f} kHz")
print(f"  Flux disp.:      {abs(w01_phi025 - w01_phi0)*1e6:.2f} kHz  (Φ₂=0 vs Φ₂=0.25)")

── Device A (toy model) ───────────────────────────
  f01 (ng=0):      4.9312 MHz
  f01 (ng=0.5):    2.2747 MHz
  Charge disp.:    2656.50 kHz
  Flux disp.:      897941.44 kHz  (Φ₂=0 vs Φ₂=0.25)


# Device B

| $E_{2J}$ | $E_S$ | $E_C$ | $E_L$ |
|----------|-------|-------|-------|
| 7.11     | 3.42  | 0.18  | 0.20  |

In [5]:
E2j_B, Es_B, Ec_B, El_B = 7.11, 3.42, 0.18, 0.20

w01_ng0  = f01_toy(Ec_B, El_B, Es_B, E2j_B, phi_ext=0.0, ng=0.0)
w01_ng05 = f01_toy(Ec_B, El_B, Es_B, E2j_B, phi_ext=0.0, ng=0.5)
w01_phi0   = f01_toy(Ec_B, El_B, Es_B, E2j_B, phi_ext=0.0,  ng=0.0)
w01_phi025 = f01_toy(Ec_B, El_B, Es_B, E2j_B, phi_ext=0.25, ng=0.0)

print("── Device B (toy model) ───────────────────────────")
print(f"  f01 (ng=0):      {w01_ng0*1e3:.4f} MHz")
print(f"  f01 (ng=0.5):    {w01_ng05*1e3:.4f} MHz")
print(f"  Charge disp.:    {abs(w01_ng05 - w01_ng0)*1e6:.2f} kHz")
print(f"  Flux disp.:      {abs(w01_phi025 - w01_phi0)*1e6:.2f} kHz  (Φ₂=0 vs Φ₂=0.25)")

── Device B (toy model) ───────────────────────────
  f01 (ng=0):      0.0008 MHz
  f01 (ng=0.5):    0.0001 MHz
  Charge disp.:    0.68 kHz
  Flux disp.:      156413.25 kHz  (Φ₂=0 vs Φ₂=0.25)


# Device C

| $E_{2J}$ | $E_S$ | $E_C$ | $E_L$ |
|----------|-------|-------|-------|
| 9.11     | 3.94  | 0.23  | 0.11  |

In [6]:
E2j_C, Es_C, Ec_C, El_C = 9.11, 3.94, 0.23, 0.11

w01_ng0  = f01_toy(Ec_C, El_C, Es_C, E2j_C, phi_ext=0.0, ng=0.0)
w01_ng05 = f01_toy(Ec_C, El_C, Es_C, E2j_C, phi_ext=0.0, ng=0.5)
w01_phi0   = f01_toy(Ec_C, El_C, Es_C, E2j_C, phi_ext=0.0,  ng=0.0)
w01_phi025 = f01_toy(Ec_C, El_C, Es_C, E2j_C, phi_ext=0.25, ng=0.0)

print("── Device C (toy model) ───────────────────────────")
print(f"  f01 (ng=0):      {w01_ng0*1e3:.4f} MHz")
print(f"  f01 (ng=0.5):    {w01_ng05*1e3:.4f} MHz")
print(f"  Charge disp.:    {abs(w01_ng05 - w01_ng0)*1e6:.2f} kHz")
print(f"  Flux disp.:      {abs(w01_phi025 - w01_phi0)*1e6:.2f} kHz  (Φ₂=0 vs Φ₂=0.25)")

── Device C (toy model) ───────────────────────────
  f01 (ng=0):      0.0009 MHz
  f01 (ng=0.5):    0.0000 MHz
  Charge disp.:    0.94 kHz
  Flux disp.:      15462.63 kHz  (Φ₂=0 vs Φ₂=0.25)
